# 03 · Memory and model selection

*Intermediate Model Training*, notebook 3 of 4 · lessons 5 and 6: **Use mixed precision and activation checkpointing** and **Design a bounded model-selection experiment**

Two kinds of budget limit every training run: memory and the number of experiments you can afford. In the first half you look at memory: what happens to numbers stored in 16 bits, how loss scaling protects small gradients, and how activation checkpointing trades extra computation for less stored data, measured on the tiny tensor engine's own tape. In the second half you spend a fixed budget of training runs on a random search, with a validation protocol that stays the same for every trial and a ledger that records all of them.

## What you will build

- `scaled_float16_gradients(grads, scale)`: the gradients a float16 backward pass would deliver after loss scaling, unscaled in float32, with an overflow flag;
- `checkpointed_backward(boundaries, layers, segment, grad_out)`: backpropagation that rebuilds one segment of the network at a time from a few stored activations;
- `sample_configuration(seed)`: one random draw from a declared search space, with a log-uniform learning rate;
- `random_search(n_trials, evaluate, first_seed=0)`: a bounded search that records every trial, good or bad, in a ledger.

## Learning objectives

By the end of this notebook you will be able to:

1. read float16's limits and predict when a value overflows, underflows to zero or is rounded away;
2. implement loss scaling, and explain why mixed-precision training keeps a float32 master copy of the weights;
3. measure the activation memory held for the backward pass, and trade it for recomputation with checkpointing;
4. run a bounded random search with a fixed validation protocol, keep a complete ledger, and report how the winner varies across seeds;
5. keep the test set out of every selection step.

### How to work through this notebook

Run the cells from top to bottom. When you reach a **✍️ Your turn** cell, type the function yourself and then run the check cell beneath it:

- ✅ means your function matches the reference, and from then on the notebook runs **your** code;
- ✍️ means you have not written it yet, and ❌ means it does not match yet. In both cases the notebook quietly uses a hidden reference version, so you can always keep going and come back later.

Each reference solution sits in a collapsed cell just above the function you type. Open it only when you are stuck.

## Dependencies

- **Tier 1:** NumPy for every calculation and Matplotlib for the plots. NumPy's `float16` type stands in for the half-precision hardware formats; nothing else is imported, so the notebook runs in JupyterLite in your browser, on the course JupyterHub server and in Google Colab.
- Nothing is downloaded: every array is generated from a fixed seed.
- The networks use the tiny tensor engine and the `MLP`, `Adam` and `fit_standardiser` from Base Model Training, plus the data generator and training helpers from notebook 1 of this course. All are pasted into hidden recap cells.
- Memory is measured as the bytes of NumPy arrays, which is exact for this engine. GPU frameworks add allocator caches, workspaces and fused kernels, so their numbers differ; the trade-offs are the same.

## Setup

Run this cell first. It imports NumPy and Matplotlib, creates a seeded random generator and defines `practice`, the helper that checks each function you write against a reference.

In [ ]:
# ── Setup ─ run this cell first ────────────────────────────────────────────
import numpy as np

rng = np.random.default_rng(0)


def _close(got, want, atol):
    if isinstance(want, dict):
        return isinstance(got, dict) and got.keys() == want.keys() and all(
            _close(got[key], want[key], atol) for key in want
        )
    if isinstance(want, (list, tuple)):
        return isinstance(got, (list, tuple)) and len(got) == len(want) and all(
            _close(g, w, atol) for g, w in zip(got, want)
        )
    try:
        got_array, want_array = np.asarray(got, dtype=float), np.asarray(want, dtype=float)
    except (TypeError, ValueError):
        return got == want
    return got_array.shape == want_array.shape and np.allclose(got_array, want_array, atol=atol)


def _short(value):
    if isinstance(value, tuple):
        return "(" + ", ".join(_short(item) for item in value) + ")"
    if isinstance(value, np.ndarray):
        text = np.array2string(value, precision=4, threshold=8, separator=", ").replace("\n", "")
    else:
        text = repr(value)
    return text if len(text) < 120 else text[:117] + "..."


def practice(name, yours, reference, cases, atol=1e-6):
    """Check your function against the reference on a few inputs.

    Returns the function the rest of the notebook should use: yours once it
    matches, otherwise the reference -- so you can always keep going.
    """
    try:
        for args in cases:
            got, want = yours(*args), reference(*args)
            if not _close(got, want, atol):
                print(f"❌ {name}: for input {_short(args)}")
                print(f"   you returned {_short(got)}")
                print(f"   expected     {_short(want)}")
                print(f"   The rest of the notebook uses the reference {name} until yours matches.")
                return reference
    except NotImplementedError:
        print(f"✍️  {name}: not written yet. Type it in the cell above and run this cell again.")
        print("   Until then the rest of the notebook uses the reference version.")
        return reference
    except Exception as error:  # a typo in the learner's code must not stop the lesson
        print(f"❌ {name} raised {type(error).__name__}: {error}")
        print(f"   The rest of the notebook uses the reference {name} until yours runs.")
        return reference
    print(f"✅ {name}: matches the reference on {len(cases)} test case{'s' if len(cases) != 1 else ''}. From here on the notebook runs YOUR {name}.")
    return yours

import time

import matplotlib.pyplot as plt

## 📦 Recap: the tiny tensor engine you built in Base Model Training

The hidden cell below is the reverse-mode autograd engine from Base Model Training, pasted unchanged: `Tensor` (with `+`, `-`, `*`, `/`, `**`, `@`, `exp`, `log`, `tanh`, `relu`, `sigmoid`, `sum`, `mean`, `reshape`, `transpose`, indexing, `log_softmax`, `softmax` and `backward`), `parameter`, `zeros_parameter`, `zero_grad`, `cross_entropy`, `layer_norm`, `numerical_grad` and `relative_error`. Every operation in it was checked against numerical gradients. Run the cell; open it only if you want to look back at the code.

In [ ]:
# ── Tiny tensor autograd (NumPy) ── the engine you built in Base Model Training ──
import numpy as np


def _unbroadcast(grad, shape):
    """Sum a gradient back down to `shape`, undoing NumPy broadcasting."""
    while grad.ndim > len(shape):
        grad = grad.sum(axis=0)
    for axis, size in enumerate(shape):
        if size == 1 and grad.shape[axis] != 1:
            grad = grad.sum(axis=axis, keepdims=True)
    return grad


class Tensor:
    """A NumPy array that remembers how it was made, so gradients can flow back."""

    def __init__(self, data, requires_grad=False, parents=(), backward=None, op=""):
        self.data = np.asarray(data, dtype=np.float64)
        self.requires_grad = requires_grad
        self.grad = None
        self._parents = parents
        self._backward = backward  # grad_out -> one gradient per parent
        self.op = op

    shape = property(lambda self: self.data.shape)
    ndim = property(lambda self: self.data.ndim)
    T = property(lambda self: self.transpose())

    def __repr__(self):
        return f"Tensor(shape={self.shape}, op={self.op or 'leaf'}, requires_grad={self.requires_grad})"

    @staticmethod
    def lift(value):
        return value if isinstance(value, Tensor) else Tensor(value)

    def _result(self, data, parents, backward, op):
        if any(parent.requires_grad for parent in parents):
            return Tensor(data, True, parents, backward, op)
        return Tensor(data, op=op)

    # arithmetic, with broadcasting
    def __add__(self, other):
        other = Tensor.lift(other)
        return self._result(self.data + other.data, (self, other), lambda g: (g, g), "add")

    __radd__ = __add__

    def __neg__(self):
        return self._result(-self.data, (self,), lambda g: (-g,), "neg")

    def __sub__(self, other):
        return self + (-Tensor.lift(other))

    def __rsub__(self, other):
        return Tensor.lift(other) + (-self)

    def __mul__(self, other):
        other = Tensor.lift(other)
        return self._result(self.data * other.data, (self, other),
                            lambda g: (g * other.data, g * self.data), "mul")

    __rmul__ = __mul__

    def __truediv__(self, other):
        other = Tensor.lift(other)
        return self._result(self.data / other.data, (self, other),
                            lambda g: (g / other.data, -g * self.data / other.data ** 2), "div")

    def __rtruediv__(self, other):
        return Tensor.lift(other) / self

    def __pow__(self, exponent):
        assert isinstance(exponent, (int, float)), "only constant exponents"
        return self._result(self.data ** exponent, (self,),
                            lambda g: (g * exponent * self.data ** (exponent - 1),), "pow")

    def __matmul__(self, other):
        other = Tensor.lift(other)

        def backward(g):
            return (g @ np.swapaxes(other.data, -1, -2), np.swapaxes(self.data, -1, -2) @ g)

        return self._result(self.data @ other.data, (self, other), backward, "matmul")

    # elementwise functions
    def exp(self):
        out = np.exp(self.data)
        return self._result(out, (self,), lambda g: (g * out,), "exp")

    def log(self):
        return self._result(np.log(self.data), (self,), lambda g: (g / self.data,), "log")

    def tanh(self):
        out = np.tanh(self.data)
        return self._result(out, (self,), lambda g: (g * (1 - out ** 2),), "tanh")

    def relu(self):
        return self._result(np.maximum(self.data, 0), (self,), lambda g: (g * (self.data > 0),), "relu")

    def sigmoid(self):
        out = 0.5 * (1 + np.tanh(0.5 * self.data))  # stable for large |x|
        return self._result(out, (self,), lambda g: (g * out * (1 - out),), "sigmoid")

    # reductions and reshaping
    def sum(self, axis=None, keepdims=False):
        def backward(g):
            if axis is not None and not keepdims:
                g = np.expand_dims(g, axis)
            return (np.broadcast_to(g, self.shape).copy(),)

        return self._result(self.data.sum(axis=axis, keepdims=keepdims), (self,), backward, "sum")

    def mean(self, axis=None, keepdims=False):
        count = self.data.size if axis is None else np.prod([self.shape[a] for a in np.atleast_1d(axis)])
        return self.sum(axis=axis, keepdims=keepdims) / count

    def reshape(self, *shape):
        return self._result(self.data.reshape(*shape), (self,), lambda g: (g.reshape(self.shape),), "reshape")

    def transpose(self, *axes):
        axes = axes or tuple(reversed(range(self.ndim)))
        inverse = np.argsort(axes)
        return self._result(self.data.transpose(axes), (self,), lambda g: (g.transpose(inverse),), "transpose")

    def __getitem__(self, index):
        def backward(g):
            full = np.zeros_like(self.data)
            np.add.at(full, index, g)  # repeated indices (an embedding row used twice) add up
            return (full,)

        return self._result(self.data[index], (self,), backward, "index")

    def log_softmax(self, axis=-1):
        shifted = self.data - self.data.max(axis=axis, keepdims=True)
        out = shifted - np.log(np.exp(shifted).sum(axis=axis, keepdims=True))
        probs = np.exp(out)
        return self._result(out, (self,), lambda g: (g - probs * g.sum(axis=axis, keepdims=True),), "log_softmax")

    def softmax(self, axis=-1):
        shifted = np.exp(self.data - self.data.max(axis=axis, keepdims=True))
        out = shifted / shifted.sum(axis=axis, keepdims=True)
        return self._result(out, (self,), lambda g: (out * (g - (g * out).sum(axis=axis, keepdims=True)),), "softmax")

    # reverse-mode differentiation
    def backward(self, grad=None):
        order, seen = [], set()

        def visit(node):
            if id(node) not in seen:
                seen.add(id(node))
                for parent in node._parents:
                    visit(parent)
                order.append(node)

        visit(self)
        self.grad = np.ones_like(self.data) if grad is None else np.asarray(grad, dtype=np.float64)
        for node in reversed(order):
            if node._backward is None or node.grad is None:
                continue
            for parent, parent_grad in zip(node._parents, node._backward(node.grad)):
                if not parent.requires_grad or parent_grad is None:
                    continue
                parent_grad = _unbroadcast(np.asarray(parent_grad, dtype=np.float64), parent.shape)
                parent.grad = parent_grad if parent.grad is None else parent.grad + parent_grad


def parameter(*shape, rng, scale=None):
    """A trainable weight, initialised with N(0, scale^2); default scale is He's sqrt(2 / fan_in)."""
    scale = np.sqrt(2.0 / shape[0]) if scale is None else scale
    return Tensor(rng.normal(0.0, scale, shape), requires_grad=True)


def zeros_parameter(*shape):
    return Tensor(np.zeros(shape), requires_grad=True)


def zero_grad(params):
    for p in params:
        p.grad = None


def cross_entropy(logits, targets, mask=None):
    """Mean next-token/class cross-entropy. logits: (..., classes); targets: int (...); mask: 1.0 keeps a position."""
    flat = logits.reshape(-1, logits.shape[-1]).log_softmax(axis=-1)
    targets = np.asarray(targets).reshape(-1)
    picked = flat[np.arange(targets.size), targets]
    if mask is None:
        return -picked.mean()
    weights = np.asarray(mask, dtype=np.float64).reshape(-1)
    return -(picked * weights).sum() / max(weights.sum(), 1.0)


def layer_norm(x, gamma, beta, eps=1e-5):
    mean = x.mean(axis=-1, keepdims=True)
    centred = x - mean
    var = (centred ** 2).mean(axis=-1, keepdims=True)
    return centred / (var + eps) ** 0.5 * gamma + beta


def numerical_grad(f, tensor, eps=1e-6):
    """Central-difference gradient of scalar f() with respect to tensor.data."""
    grad = np.zeros_like(tensor.data)
    it = np.nditer(tensor.data, flags=["multi_index"])
    for _ in it:
        index = it.multi_index
        original = tensor.data[index]
        tensor.data[index] = original + eps
        plus = float(f().data)
        tensor.data[index] = original - eps
        minus = float(f().data)
        tensor.data[index] = original
        grad[index] = (plus - minus) / (2 * eps)
    return grad


def relative_error(a, b):
    return float(np.linalg.norm(a - b) / max(np.linalg.norm(a) + np.linalg.norm(b), 1e-12))

## 📦 Recap: what you built earlier

The second hidden cell re-defines, under their usual names, the pieces from Base Model Training (`fit_standardiser`, `Linear`, `dropout`, `MLP`, `Adam`) and from notebook 1 of this course: `make_rare_class_data(n, seed)`, the no-skill baseline `prevalence_log_loss(y_train, y_eval)`, `mean_loss(model, X, y)` and `fit(model, X, y, steps, lr, batch_size, seed)`, which trains with Adam on seeded mini-batches.

In [ ]:
# 📦 Recap: the pieces from Base Model Training and notebook 1, under their usual names.

def fit_standardiser(X):
    """Column means and scales of the TRAINING rows; apply them to every split as (X - mean) / scale."""
    return X.mean(axis=0), np.maximum(X.std(axis=0), 1e-6)


class Linear:
    """A dense layer x @ W + b, with He-initialised W of shape (n_in, n_out) and a zero bias."""

    def __init__(self, n_in, n_out, rng):
        self.W = parameter(n_in, n_out, rng=rng)
        self.b = zeros_parameter(n_out)

    def __call__(self, x):
        return x @ self.W + self.b

    def parameters(self):
        return [self.W, self.b]


def dropout(x, p, training, rng):
    """Inverted dropout: while training, zero each unit with probability p and scale the rest by 1 / (1 - p)."""
    if not training or p == 0.0:
        return x
    keep = (rng.random(x.shape) >= p) / (1.0 - p)
    return x * keep


class MLP:
    """ReLU hidden layers and a linear output layer, e.g. MLP([6, 32, 2], rng). Call it on a Tensor (batch, n_in)."""

    def __init__(self, sizes, rng, dropout=0.0):
        self.layers = [Linear(n_in, n_out, rng) for n_in, n_out in zip(sizes[:-1], sizes[1:])]
        self.p = dropout
        self.training = True

    def __call__(self, x, rng=None):
        for index, layer in enumerate(self.layers):
            x = layer(x)
            if index < len(self.layers) - 1:
                x = dropout(x.relu(), self.p, self.training, rng)
        return x

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]

    def train(self):
        self.training = True

    def eval(self):
        self.training = False


class Adam:
    """Adam over a list of engine parameters: opt.zero_grad(), loss.backward(), opt.step()."""

    def __init__(self, params, lr=0.01, beta1=0.9, beta2=0.999, eps=1e-8):
        self.params, self.lr, self.beta1, self.beta2, self.eps = list(params), lr, beta1, beta2, eps
        self.m = [np.zeros_like(p.data) for p in self.params]
        self.v = [np.zeros_like(p.data) for p in self.params]
        self.t = 0

    def zero_grad(self):
        zero_grad(self.params)

    def step(self):
        self.t += 1
        for p, m, v in zip(self.params, self.m, self.v):
            if p.grad is None:
                continue
            m[...] = self.beta1 * m + (1 - self.beta1) * p.grad
            v[...] = self.beta2 * v + (1 - self.beta2) * p.grad ** 2
            m_hat = m / (1 - self.beta1 ** self.t)
            v_hat = v / (1 - self.beta2 ** self.t)
            p.data -= self.lr * m_hat / (np.sqrt(v_hat) + self.eps)


# From notebook 1 of this course.

def make_rare_class_data(n, seed):
    """n rows of six standard-normal features; label 1 when a noisy score exceeds 2.8 (about 11% of rows)."""
    g = np.random.default_rng(seed)
    X = g.normal(size=(n, 6))
    score = 1.5 * X[:, 0] - X[:, 1] + X[:, 2] * X[:, 3] + 0.8 * g.normal(size=n)
    return X, (score > 2.8).astype(int)


def prevalence_log_loss(y_train, y_eval):
    """Log loss of always predicting the training prevalence: the no-skill baseline."""
    pi = np.clip(np.mean(y_train), 1e-12, 1 - 1e-12)
    return float(-np.mean(y_eval * np.log(pi) + (1 - y_eval) * np.log(1 - pi)))


def mean_loss(model, X, y):
    """Mean cross-entropy of a model on (X, y), in evaluation mode."""
    model.eval()
    return float(cross_entropy(model(Tensor(X)), y).data)


def fit(model, X, y, steps=300, lr=0.01, batch_size=128, seed=0, loss_fn=cross_entropy):
    """Train with Adam on random mini-batches drawn with a seeded generator; returns the model in eval mode."""
    opt = Adam(model.parameters(), lr=lr)
    g = np.random.default_rng(seed)
    for _ in range(steps):
        rows = g.integers(0, len(y), size=min(batch_size, len(y)))
        model.train()
        opt.zero_grad()
        loss = loss_fn(model(Tensor(X[rows]), g), y[rows])
        loss.backward()
        opt.step()
    model.eval()
    return model

## 1 · What 16 bits can hold

A float16 number has 1 sign bit, 5 exponent bits and 10 fraction bits, half the storage of float32. That halves the memory and bandwidth for every array stored in it, which is why accelerators compute in 16-bit formats where they safely can. The price is range and precision:

- the largest float16 is 65,504; anything larger becomes `inf` (**overflow**);
- below about $6.1 \times 10^{-5}$ the numbers become *subnormal* and lose precision, and anything below about $3 \times 10^{-8}$ becomes 0 (**underflow**);
- the spacing between neighbouring numbers is $2^{-10} \approx 0.001$ relative to their size, so adding something smaller than half that spacing changes nothing.

The cell shows all three, and what happens to a running total kept in float16.

In [ ]:
info = np.finfo(np.float16)
print(f"largest float16 {info.max}, smallest normal {info.tiny}, smallest subnormal {info.smallest_subnormal}, epsilon {info.eps}")
with np.errstate(over="ignore"):
    print("70000 as float16:", float(np.float16(70000.0)), "   1e-8 as float16:", float(np.float16(1e-8)),
          "   1e-6 as float16:", float(np.float16(1e-6)))
print("2048 + 1 in float16:", float(np.float16(2048) + np.float16(1)), "   1 + 0.0001 in float16:", float(np.float16(1) + np.float16(1e-4)))

total16, total32 = np.float16(0.0), np.float32(0.0)
for _ in range(5000):
    total16 = np.float16(total16 + np.float16(0.01))
    total32 = np.float32(total32 + np.float32(0.01))
print(f"adding 0.01 five thousand times: float16 total {total16}, float32 total {total32:.3f}, exact 50")

**What to notice.** 70,000 overflows to `inf` and $10^{-8}$ underflows to 0, while $10^{-6}$ survives only as a coarse subnormal, stored as about $1.013 \times 10^{-6}$. $2048 + 1$ gives 2048, and $1 + 0.0001$ gives exactly 1. The float16 running total gets stuck at 32: from there on, 0.01 is less than half the spacing between neighbouring float16 numbers, so every addition rounds back. This is why frameworks accumulate sums in float32 even when the inputs are float16.

The same rounding attacks training. A weight of 1.0 receiving updates of 0.0002 never moves if the weight itself is stored in float16. Mixed-precision training therefore keeps a float32 **master copy** of the weights, applies the updates to it, and only uses float16 copies for the arithmetic.

In [ ]:
target, lr = 1.2, 0.001
w16, w32 = np.float16(1.0), np.float32(1.0)
for _ in range(2000):
    w16 = np.float16(w16 - np.float16(lr) * (w16 - np.float16(target)))  # the weight itself stored in float16
    w32 = np.float32(w32 - np.float32(lr) * (w32 - np.float32(target)))  # a float32 master copy
print(f"after 2000 small updates towards {target}: float16 weight {float(w16):.4f}, float32 master weight {float(w32):.4f}")

## 2 · Small gradients underflow, and loss scaling rescues them

Gradients in a deep network span many orders of magnitude, and the smallest are often the first casualties of float16. The cell builds three made-up gradient arrays with typical sizes $10^{-2}$, $10^{-5}$ and $10^{-8}$, as if from an output layer, a middle layer and a first layer, and stores them in float16.

In [ ]:
g = np.random.default_rng(0)
layer_names = ["output layer", "middle layer", "first layer"]
layer_grads = [g.normal(size=500) * 1e-2, g.normal(size=500) * 1e-5, g.normal(size=500) * 1e-8]
for name, grad in zip(layer_names, layer_grads):
    stored = grad.astype(np.float16).astype(np.float64)
    print(f"{name:>12}: typical size {np.abs(grad).mean():.0e}, flushed to zero {np.mean(stored == 0):4.0%}, "
          f"relative error {np.linalg.norm(stored - grad) / np.linalg.norm(grad):.1e}")

The first layer's gradient is almost entirely lost: most of its entries become exactly 0, and that layer would stop learning. **Loss scaling** fixes this with one multiplication. Multiply the loss by a large factor $S$ before the backward pass; by the chain rule every gradient is multiplied by $S$ too, which lifts the small ones into float16's range. After the backward pass, convert the gradients to float32 and divide by $S$:

$$g_{\text{used}} = \frac{\operatorname{float32}\big(\operatorname{float16}(S \cdot g)\big)}{S}.$$

If $S$ is too large, the *largest* gradients overflow to `inf`. The update must then be skipped and $S$ lowered. Frameworks such as PyTorch automate this as **dynamic loss scaling**: start with a large $S$ such as $2^{16}$, halve it and skip the update whenever an overflow appears, and double it again after a long run of clean updates. Exercise 1 implements that rule.

**Shapes.** `grads` is a list of float64 arrays of any shapes, the true gradients of the *unscaled* loss. The function returns a list of float32 arrays of the same shapes and one `bool`.

### ✍️ Your turn: `scaled_float16_gradients`

Write `scaled_float16_gradients(grads, scale)` that simulates one scaled float16 backward pass:

- **inputs:** `grads`, a list of float64 arrays; `scale`, the loss scale $S$;
- **output:** `(unscaled, found_inf)`: for each array, `(g * scale).astype(np.float16)` is what float16 would store; convert that to `np.float32` and divide by `np.float32(scale)`. `found_inf` is `True` if any stored value is not finite.

Hints: wrap the cast in `with np.errstate(over="ignore"):` so that an overflow does not print a warning (you detect it yourself with `np.isfinite`); `bool(...)` turns a NumPy boolean into a Python one.

*Type it rather than pasting it: typing is how the shapes and indices sink in.* The reference solution is in the collapsed cell below; open it only if you are stuck.

In [ ]:
def scaled_float16_gradients_reference(grads, scale):
    with np.errstate(over="ignore"):
        stored = [(g * scale).astype(np.float16) for g in grads]
    found_inf = not all(bool(np.all(np.isfinite(s))) for s in stored)
    return [s.astype(np.float32) / np.float32(scale) for s in stored], found_inf

In [ ]:
def scaled_float16_gradients(grads, scale):
    """Gradients after a float16 backward pass with loss scale `scale`, unscaled in float32.
    grads: list of float64 arrays. Returns (list of float32 arrays, found_inf)."""
    # ✍️ Your code here (about four lines)
    raise NotImplementedError

In [ ]:
scaled_float16_gradients = practice("scaled_float16_gradients", scaled_float16_gradients, scaled_float16_gradients_reference, cases=[
    ([np.array([1e-8, -3e-8, 2e-3])], 1.0),
    ([np.array([1e-8, -3e-8, 2e-3]), np.array([[0.5, -1e-6]])], 2.0 ** 16),
    ([np.array([1e-2, 5.0])], 2.0 ** 16),  # edge case: 5 * 65536 overflows
], atol=0.0)

print(f"{'scale':>7} | " + " | ".join(f"{name:>12}" for name in layer_names) + " | overflow")
for exponent in [0, 8, 16, 24]:
    unscaled, found_inf = scaled_float16_gradients(layer_grads, 2.0 ** exponent)
    errors = [np.linalg.norm(u - g) / np.linalg.norm(g) for u, g in zip(unscaled, layer_grads)]
    print(f"{'2^' + str(exponent):>7} | " + " | ".join(f"{e:12.1e}" for e in errors) + f" | {found_inf}")

**Expected behaviour.** With scale 1 the first layer's relative error is about 1: its gradient is essentially gone. With $2^{16}$ all three layers come back with errors of about $10^{-3}$ or less, the precision float16 can offer. With $2^{24}$ the output layer's gradients exceed 65,504 and overflow, so `found_inf` is `True` and this update would be skipped. (The check passes `atol=0.0` to `practice` because these gradients are tiny: the default tolerance of $10^{-6}$ would call a gradient of $10^{-8}$ equal to zero.)

**bfloat16**, the other common 16-bit format, keeps float32's 8 exponent bits and only 7 fraction bits. It almost never underflows, so it rarely needs loss scaling, but it rounds even more coarsely than float16 (Exercise 2 simulates it). Which format is faster depends on the hardware, so check what your accelerator supports and compare a short run against a float32 reference before trusting either.

## 3 · Activation memory, and checkpointing

During the forward pass, the tape keeps every intermediate array that the backward pass will need: each layer's matrix product, its sum with the bias and its activation. For a deep network on a large batch these **activations** can need far more memory than the weights. **Activation checkpointing** keeps only a few of them, the *boundaries* between segments of $k$ layers, and rebuilds each segment from its boundary during the backward pass:

1. forward pass without a tape, keeping only $h_0, h_k, h_{2k}, \dots$ and the output;
2. compute the gradient of the loss with respect to the output;
3. for each segment, from the last to the first: rebuild its graph from its boundary, backpropagate the incoming gradient through it, and pass the gradient at its input on to the previous segment.

Every layer is evaluated twice, once in each pass, so checkpointing costs roughly one extra forward pass. In return, only the boundaries plus one segment's tape are ever held at once. The engine's `backward(grad)` accepts an incoming gradient, which is exactly what step 3 needs.

The network below has 12 layers of width 64, each $h \mapsto \tanh(hW + b)$, on a batch of 256 rows. `TanhLayer.calls` counts layer evaluations, and `tape_bytes` walks the graph from a result and adds up the bytes of every array it keeps alive, parameters excluded.

In [ ]:
WIDTH, DEPTH, BATCH = 64, 12, 256


class TanhLayer:
    """One layer h -> tanh(h @ W + b). TanhLayer.calls counts every evaluation, recorded on the tape or not."""
    calls = 0

    def __init__(self, width, rng):
        self.W = parameter(width, width, rng=rng, scale=1.0 / np.sqrt(width))
        self.b = zeros_parameter(width)

    def __call__(self, h):  # Tensor -> Tensor, recorded on the tape
        TanhLayer.calls += 1
        return (h @ self.W + self.b).tanh()

    def apply(self, h):  # array -> array, nothing recorded
        TanhLayer.calls += 1
        return np.tanh(h @ self.W.data + self.b.data)


def run_layers(h, layers):
    for layer in layers:
        h = layer(h)
    return h


def tape_bytes(result, params):
    """Bytes of every array that the graph behind `result` keeps alive for backward, parameters excluded."""
    skip = {id(p) for p in params}
    seen, stack, total = set(), [result], 0
    while stack:
        node = stack.pop()
        if id(node) in seen:
            continue
        seen.add(id(node))
        total += 0 if id(node) in skip else node.data.nbytes
        stack.extend(node._parents)
    return total


g = np.random.default_rng(1)
deep = [TanhLayer(WIDTH, g) for _ in range(DEPTH)]
deep_params = [p for layer in deep for p in (layer.W, layer.b)]
X_deep, Y_deep = g.normal(size=(BATCH, WIDTH)), g.normal(size=(BATCH, WIDTH))
loss_head = lambda out: ((out - Y_deep) ** 2).mean()

TanhLayer.calls = 0
zero_grad(deep_params)
loss = loss_head(run_layers(Tensor(X_deep), deep))
full_tape = tape_bytes(loss, deep_params)
loss.backward()
full_grads = [p.grad.copy() for p in deep_params]
full_calls = TanhLayer.calls
weight_bytes = sum(p.data.nbytes for p in deep_params)
print(f"weights: {weight_bytes / 1e6:.2f} MB; activations kept on the tape: {full_tape / 1e6:.2f} MB; layer evaluations: {full_calls}")

**What to notice.** The activations kept for the backward pass take more than ten times the memory of the weights here, and they grow with the batch size and the depth, while the weights do not. The forward pass that keeps only boundaries is given: it evaluates every layer on plain arrays with `apply`, so nothing is recorded.

In [ ]:
def forward_keep_boundaries(X, layers, segment):
    """Run every layer on plain arrays (no tape) and keep only h_0, h_k, h_2k, ... and the output."""
    kept, h = [X], X
    for index, layer in enumerate(layers):
        h = layer.apply(h)
        if (index + 1) % segment == 0 or index + 1 == len(layers):
            kept.append(h)
    return kept


boundaries_demo = forward_keep_boundaries(X_deep, deep, 3)
print("boundaries kept with segments of 3:", len(boundaries_demo), "arrays of shape", boundaries_demo[0].shape)

### ✍️ Your turn: `checkpointed_backward`

Write `checkpointed_backward(boundaries, layers, segment, grad_out)`, step 3 above:

- **inputs:** `boundaries` from `forward_keep_boundaries` (arrays $h_0, h_k, \dots$, output last); the list of `layers`; the segment length `segment`; and `grad_out`, the gradient of the loss with respect to the output, an array of the output's shape;
- **output:** the gradient with respect to the network's input $h_0$, an array. As a side effect, every layer's `W.grad` and `b.grad` receive their gradients (the caller zeroes them first).

Steps: loop `s` over `reversed(range(len(boundaries) - 1))`; make `start = Tensor(boundaries[s], requires_grad=True)`; rebuild the segment with `run_layers(start, layers[s * segment:(s + 1) * segment])`; call `.backward(grad_out)` on its output; then set `grad_out = start.grad` for the next segment down. Return `grad_out` at the end.

*Type it rather than pasting it: typing is how the shapes and indices sink in.* The reference is in the collapsed cell below.

In [ ]:
def checkpointed_backward_reference(boundaries, layers, segment, grad_out):
    for s in reversed(range(len(boundaries) - 1)):
        start = Tensor(boundaries[s], requires_grad=True)
        out = run_layers(start, layers[s * segment:(s + 1) * segment])
        out.backward(grad_out)
        grad_out = start.grad
    return grad_out

In [ ]:
def checkpointed_backward(boundaries, layers, segment, grad_out):
    """Backpropagate grad_out through the network one rebuilt segment at a time, last segment first.
    Fills the layers' parameter gradients; returns the gradient with respect to boundaries[0]."""
    # ✍️ Your code here (about five lines)
    raise NotImplementedError

The check rebuilds fresh gradients for both versions and compares the input gradient **and** every parameter gradient, on a small 5-layer network with segments of 2, 2 and 1 layers, with a boundary after every layer, and with a single segment.

In [ ]:
def gradients_after(backward_function):
    """Zero the parameter gradients, run a checkpointed backward, and return the input and parameter gradients."""
    def run(boundaries, layers, segment, grad_out):
        params = [p for layer in layers for p in (layer.W, layer.b)]
        zero_grad(params)
        grad_in = backward_function(boundaries, layers, segment, grad_out)
        return [grad_in] + [p.grad.copy() for p in params]
    return run


small_net = [TanhLayer(4, np.random.default_rng(2)) for _ in range(5)]
X_small, upstream = np.random.default_rng(3).normal(size=(3, 4)), np.random.default_rng(4).normal(size=(3, 4))
yours = gradients_after(checkpointed_backward)
checked = practice("checkpointed_backward", yours, gradients_after(checkpointed_backward_reference), cases=[
    (forward_keep_boundaries(X_small, small_net, 2), small_net, 2, upstream),  # segments of 2, 2 and 1 layers
    (forward_keep_boundaries(X_small, small_net, 1), small_net, 1, upstream),  # a boundary after every layer
    (forward_keep_boundaries(X_small, small_net, 5), small_net, 5, upstream),  # edge case: one segment
])
checkpointed_backward = checkpointed_backward if checked is yours else checkpointed_backward_reference

Now the measurement on the 12-layer network: the gradients, the bytes kept, and the number of layer evaluations, with segments of 3 layers. The experiment in Section 5 tries other lengths.

In [ ]:
def checkpointed_gradients(X, layers, segment, loss_head):
    """Gradients of loss_head(network(X)) with checkpointing. Returns (parameter gradients, boundaries kept)."""
    params = [p for layer in layers for p in (layer.W, layer.b)]
    zero_grad(params)
    boundaries = forward_keep_boundaries(X, layers, segment)       # 1. forward without a tape
    top = Tensor(boundaries[-1], requires_grad=True)
    loss_head(top).backward()                                      # 2. gradient at the output
    checkpointed_backward(boundaries, layers, segment, top.grad)   # 3. rebuild and backpropagate, segment by segment
    return [p.grad.copy() for p in params], boundaries


def measure_checkpointing(segment):
    TanhLayer.calls = 0
    grads, boundaries = checkpointed_gradients(X_deep, deep, segment, loss_head)
    calls = TanhLayer.calls
    kept = sum(b.nbytes for b in boundaries)
    one_segment = tape_bytes(run_layers(Tensor(boundaries[0], requires_grad=True), deep[:segment]), deep_params)
    error = max(relative_error(a, b) for a, b in zip(grads, full_grads))
    return {"kept": kept, "peak": kept + one_segment, "calls": calls, "error": error}


def best_time(function, repeats=5):
    best = np.inf
    for _ in range(repeats):
        start = time.perf_counter()
        function()
        best = min(best, time.perf_counter() - start)
    return best


checkpoint = measure_checkpointing(3)
print(f"full tape:      {full_tape / 1e6:.2f} MB kept for backward, {full_calls} layer evaluations")
print(f"checkpointing:  {checkpoint['kept'] / 1e6:.2f} MB kept between the passes, about {checkpoint['peak'] / 1e6:.2f} MB "
      f"at the peak (boundaries plus one rebuilt segment), {checkpoint['calls']} layer evaluations")
print(f"largest relative difference from the full-tape gradients: {checkpoint['error']:.1e}")
full_time = best_time(lambda: loss_head(run_layers(Tensor(X_deep), deep)).backward())
checkpoint_time = best_time(lambda: checkpointed_gradients(X_deep, deep, 3, loss_head))
print(f"time per gradient: full tape {full_time * 1e3:.1f} ms, checkpointed {checkpoint_time * 1e3:.1f} ms "
      f"(ratio {checkpoint_time / full_time:.2f}; timings vary from run to run)")

**Expected behaviour.** The gradients agree with the full-tape ones exactly, or to rounding error on other machines: checkpointing changes *when* activations are computed, not *what* is computed. Between the passes it keeps about an eighth of the full tape, and even at its peak it holds well under half. In exchange it evaluates every layer twice, 24 evaluations instead of 12, and takes somewhat longer. Checkpointing reduces activation memory only: the weights, their gradients and the optimiser's state still need their full space. Recomputed segments must behave exactly as in the first pass, so layers with randomness, such as dropout, need their random state replayed; that is why frameworks such as PyTorch provide a supported checkpoint function rather than leaving it to hand-written code.

## 4 · A bounded random search

Now a different budget: the number of training runs. We want a good configuration for the rare-class network from notebook 1, and we decide the whole **protocol** before the first run:

- **data:** the same training, validation and test splits as in notebook 1, standardised with training statistics;
- **metric:** validation log loss; the test set is not used for any choice;
- **budget:** 12 trials of 300 Adam updates each, all with the same training seed;
- **search space:** learning rate log-uniform between $10^{-3.5}$ and $10^{-0.5}$; hidden width one of 4, 8, 16, 32 or 64; dropout uniform between 0 and 0.5.

The learning rate is drawn on a **log scale** because its effect is multiplicative: the interval from 0.001 to 0.01 matters as much as the one from 0.01 to 0.1. Random search draws each setting independently, so a fixed number of trials explores every dimension, unlike a grid that repeats the same few values of the unimportant settings.

`evaluate(config)` is the protocol in code: it trains one model and returns its measurements.

In [ ]:
X_train_raw, y_train = make_rare_class_data(1000, seed=11)
X_val_raw, y_val = make_rare_class_data(500, seed=22)
X_test_raw, y_test = make_rare_class_data(500, seed=33)
mean, scale = fit_standardiser(X_train_raw)
X_train, X_val, X_test = [(X - mean) / scale for X in (X_train_raw, X_val_raw, X_test_raw)]
baseline = prevalence_log_loss(y_train, y_val)


def evaluate(config, seed=0, updates=300):
    """The fixed protocol: train on the training split with `config`, measure on the validation split."""
    start = time.perf_counter()
    model = MLP([6, config["width"], 2], np.random.default_rng(seed), dropout=config["dropout"])
    fit(model, X_train, y_train, steps=updates, lr=config["lr"], seed=seed + 1)
    val_loss = mean_loss(model, X_val, y_val)
    return {"val_loss": val_loss if np.isfinite(val_loss) else np.inf, "seconds": time.perf_counter() - start}


print(f"no-skill validation loss: {baseline:.3f}")
print("one trial:", evaluate({"lr": 0.01, "width": 32, "dropout": 0.1}))

### ✍️ Your turn: `sample_configuration`

Write `sample_configuration(seed)` that draws one configuration from the search space:

- **input:** an integer `seed`; create `g = np.random.default_rng(seed)` inside the function, so that the same seed always gives the same configuration;
- **output:** a dict `{"lr": float, "width": int, "dropout": float}`, drawn **in this order**: `lr = 10 ** g.uniform(-3.5, -0.5)`, then `width = g.choice([4, 8, 16, 32, 64])`, then `dropout = g.uniform(0.0, 0.5)`.

The order matters, because each draw consumes random numbers from the same generator. Convert with `float(...)` and `int(...)` so that the ledger holds plain Python numbers.

*Type it rather than pasting it: typing is how the shapes and indices sink in.* The reference is in the collapsed cell below.

In [ ]:
def sample_configuration_reference(seed):
    g = np.random.default_rng(seed)
    lr = 10 ** g.uniform(-3.5, -0.5)
    width = g.choice([4, 8, 16, 32, 64])
    dropout = g.uniform(0.0, 0.5)
    return {"lr": float(lr), "width": int(width), "dropout": float(dropout)}

In [ ]:
def sample_configuration(seed):
    """One draw from the search space: log-uniform lr, a width from a list, uniform dropout. Returns a dict."""
    # ✍️ Your code here (about five lines)
    raise NotImplementedError

In [ ]:
sample_configuration = practice("sample_configuration", sample_configuration, sample_configuration_reference, cases=[
    (0,), (1,), (12345,),
])
print([sample_configuration(seed) for seed in range(3)])

**Expected behaviour.** The same seed always returns the same dict, and different seeds spread the learning rate over three orders of magnitude.

### ✍️ Your turn: `random_search`

Write `random_search(n_trials, evaluate, first_seed=0)`:

- **inputs:** the budget `n_trials`; the protocol `evaluate(config)`, which returns a dict of measurements; the first configuration seed;
- **output:** the **ledger**, a list with one dict per trial, in the order run: `{"trial": t, "seed": first_seed + t}`, then the configuration's keys, then the measurements' keys. Record *every* trial, including the bad ones.

Hints: `config = sample_configuration(first_seed + trial)`; build each row with `row = {"trial": trial, "seed": first_seed + trial}`, then `row.update(config)` and `row.update(evaluate(config))`.

*Type it rather than pasting it: typing is how the shapes and indices sink in.* The reference is in the collapsed cell below.

In [ ]:
def random_search_reference(n_trials, evaluate, first_seed=0):
    ledger = []
    for trial in range(n_trials):
        config = sample_configuration(first_seed + trial)
        row = {"trial": trial, "seed": first_seed + trial}
        row.update(config)
        row.update(evaluate(config))
        ledger.append(row)
    return ledger

In [ ]:
def random_search(n_trials, evaluate, first_seed=0):
    """Evaluate n_trials sampled configurations; return the ledger: one dict per trial, every trial kept."""
    ledger = []
    # ✍️ Your code here (about six lines)
    raise NotImplementedError

In [ ]:
quick_protocol = lambda config: {"val_loss": (np.log10(config["lr"]) + 2) ** 2 + config["dropout"]}  # instant, for the check
random_search = practice("random_search", random_search, random_search_reference, cases=[
    (3, quick_protocol, 0),
    (2, quick_protocol, 40),
    (0, quick_protocol, 0),  # edge case: no budget, empty ledger
])

Now the real search: 12 trials under the fixed protocol. The ledger is printed in full, sorted by validation loss, and trials that do worse than the no-skill baseline are marked rather than hidden.

In [ ]:
start = time.perf_counter()
ledger = random_search(12, evaluate, first_seed=0)
print(f"12 trials in {time.perf_counter() - start:.1f}s; no-skill validation loss {baseline:.3f}\n")
print(f"{'trial':>5} {'lr':>8} {'width':>5} {'dropout':>7} {'val loss':>8} {'seconds':>7}")
for row in sorted(ledger, key=lambda r: r["val_loss"]):
    flag = "  <- worse than no skill" if row["val_loss"] > baseline else ""
    print(f"{row['trial']:5d} {row['lr']:8.4f} {row['width']:5d} {row['dropout']:7.2f} {row['val_loss']:8.3f} {row['seconds']:7.2f}{flag}")
best_row = min(ledger, key=lambda r: r["val_loss"])

**What to notice.** The validation losses spread widely: the worst trials, usually with tiny learning rates or tiny widths, barely beat no skill or fail to, while the best are close to the loss the MLP reached in notebook 1. A ledger that kept only its top row would hide how sensitive the result is to these choices.

### Is the winner really the best? Repeat it across seeds

The best of 12 trials is partly the best of 12 *draws of luck*: its validation loss is an optimistic estimate of how that configuration performs. Re-train the three best configurations with three new training seeds each, on the same splits, and compare the averages. The test set is still untouched.

In [ ]:
CONFIG_KEYS = ("lr", "width", "dropout")
repeats = []
for row in sorted(ledger, key=lambda r: r["val_loss"])[:3]:
    config = {key: row[key] for key in CONFIG_KEYS}
    losses = [row["val_loss"]] + [evaluate(config, seed=s)["val_loss"] for s in (1, 2, 3)]
    repeats.append((config, np.array(losses)))
    print(f"trial {row['trial']:2d}: search score {losses[0]:.3f}; over 4 seeds mean {np.mean(losses):.3f}, "
          f"std {np.std(losses, ddof=1):.3f}, range {min(losses):.3f} to {max(losses):.3f}")
chosen_config, chosen_losses = min(repeats, key=lambda item: item[1].mean())
print("chosen by the mean over seeds:", chosen_config)

The spread across seeds is often as large as the gaps between the top configurations, so ranking them by a single run is unreliable. Choosing by the mean over seeds costs more runs but gives a fairer estimate; report the spread with it.

Only now, with every choice made, does the test set get its single look.

In [ ]:
final_model = MLP([6, chosen_config["width"], 2], np.random.default_rng(0), dropout=chosen_config["dropout"])
fit(final_model, X_train, y_train, steps=300, lr=chosen_config["lr"], seed=1)
test_loss = mean_loss(final_model, X_test, y_test)
test_baseline = prevalence_log_loss(y_train, y_test)
print(f"test log loss {test_loss:.3f} (no-skill on test {test_baseline:.3f}); validation mean over seeds {chosen_losses.mean():.3f}")

## 5 · 🧪 Experiments

### 🧪 Choose the loss scale

Pick a scale and see which layers survive. Try values from $2^0$ to $2^{24}$ and find the range in which no layer is lost and nothing overflows.

In [ ]:
loss_scale = 2.0 ** 16  # 🧪 try 1.0, 2.0 ** 8, 2.0 ** 20 and 2.0 ** 24
unscaled, found_inf = scaled_float16_gradients(layer_grads, loss_scale)
for name, u, grad in zip(layer_names, unscaled, layer_grads):
    print(f"{name:>12}: relative error {np.linalg.norm(u - grad) / np.linalg.norm(grad):.1e}, zeros {np.mean(u == 0):4.0%}")
print("overflow, so skip this update and halve the scale" if found_inf else "no overflow: apply the update")

### 🧪 How long should a segment be?

For each segment length, the cell reports the bytes kept between the passes, the peak while one segment is rebuilt, and the layer evaluations. Long segments keep few boundaries but rebuild a big tape; short segments do the opposite. If a boundary costs one array and each rebuilt layer costs $c$ arrays on the tape, the peak is about $(L/k + 1) + c\,k$ arrays for $L$ layers, smallest near $k = \sqrt{L/c}$. Here every layer keeps $c = 3$ arrays (product, sum and tanh), so $k = 2$ should win narrowly over 3.

In [ ]:
segments_to_try = [1, 2, 3, 4, 6, 12]  # 🧪 add or remove lengths
print(f"full tape: {full_tape / 1e6:.2f} MB, {full_calls} layer evaluations")
for k in segments_to_try:
    m = measure_checkpointing(k)
    print(f"segment {k:2d}: kept {m['kept'] / 1e6:.2f} MB, peak {m['peak'] / 1e6:.2f} MB, {m['calls']} evaluations, "
          f"gradient difference {m['error']:.0e}")

### 🧪 A smaller or larger budget

Run a fresh search with different configuration seeds and another budget. How much does the best validation loss improve from 4 to 24 trials, and how far is the best search score from that configuration's own mean over seeds?

In [ ]:
budget = 4  # 🧪 try 12 and 24 (each trial takes a fraction of a second)
other = random_search(budget, evaluate, first_seed=1000)
top = min(other, key=lambda r: r["val_loss"])
again = [evaluate({key: top[key] for key in CONFIG_KEYS}, seed=s)["val_loss"] for s in (1, 2, 3)]
print(f"budget {budget}: best search score {top['val_loss']:.3f}; the same configuration over three new seeds: "
      f"mean {np.mean(again):.3f}")

On this small problem a wide region of the space reaches a validation loss of about 0.12, so extra trials improve the best score little or not at all. On harder problems the good region is narrower and the budget matters more. Compare each winner's search score with its re-run mean, too: the best of many noisy measurements is partly luck, so with many trials the winner's single score tends to be optimistic. Here the seed-to-seed noise is small and so is the gap; on noisier problems it can be large. Successive halving and similar schemes stop poor trials early to afford more of them, at the risk of discarding settings that start slowly.

## 6 · Put it together

The cell checks behaviour rather than exact numbers: float16's limits and the master-weight effect, loss scaling, the checkpointing trade-off measured on the tape, and the search protocol.

In [ ]:
with np.errstate(over="ignore"):
    assert np.isinf(np.float16(70000.0)) and np.float16(1e-8) == 0 and np.float16(1) + np.float16(1e-4) == 1
assert float(w16) == 1.0 and float(w32) > 1.1, "float16 weights stall; a float32 master copy keeps learning"
errors_at = {}
for exponent in (0, 16, 24):
    unscaled, found_inf = scaled_float16_gradients(layer_grads, 2.0 ** exponent)
    errors_at[exponent] = ([np.linalg.norm(u - g) / np.linalg.norm(g) for u, g in zip(unscaled, layer_grads)], found_inf)
assert errors_at[0][0][2] > 0.5 and not errors_at[0][1], "without scaling, the first layer's gradient is lost"
assert max(errors_at[16][0]) < 1e-2 and not errors_at[16][1], "a scale of 2**16 should rescue every layer"
assert errors_at[24][1], "a scale of 2**24 should overflow"
assert checkpoint["error"] < 1e-10, "checkpointing must reproduce the full-tape gradients"
assert checkpoint["kept"] < 0.3 * full_tape and checkpoint["peak"] < 0.5 * full_tape, "checkpointing should save memory"
assert checkpoint["calls"] == 2 * DEPTH and full_calls == DEPTH, "each layer is evaluated once per pass"
assert len(ledger) == 12 and all("val_loss" in row and "lr" in row for row in ledger), "the ledger keeps every trial"
assert best_row["val_loss"] < 0.6 * baseline, "the best trial should beat the no-skill baseline clearly"
assert max(r["val_loss"] for r in ledger) > 1.3 * best_row["val_loss"], "the search space should matter"
assert chosen_losses.mean() < 0.6 * baseline and test_loss < 0.6 * test_baseline, "the chosen model should hold up on test"
print("✅ Float16 limits, loss scaling, checkpointing and the bounded search all behave as expected.")
print(f"   checkpointing kept {checkpoint['kept'] / full_tape:.0%} of the full tape between passes, "
      f"peak {checkpoint['peak'] / full_tape:.0%}, for {checkpoint['calls']} instead of {full_calls} layer evaluations")
print(f"   search: best validation loss {best_row['val_loss']:.3f}; chosen configuration {chosen_losses.mean():.3f} "
      f"± {np.std(chosen_losses, ddof=1):.3f} over 4 seeds; test {test_loss:.3f}")

## Exercises

### Exercise 1: dynamic loss scaling

Write `update_loss_scale(scale, found_inf, good_steps, growth_interval=2000)`, the rule frameworks use between updates. It returns `(new_scale, new_good_steps, apply_update)`:

- if `found_inf`: halve the scale, reset `good_steps` to 0 and do **not** apply the update;
- otherwise apply the update and add 1 to `good_steps`; when that reaches `growth_interval`, double the scale and reset `good_steps` to 0.

In [ ]:
def update_loss_scale_reference(scale, found_inf, good_steps, growth_interval=2000):
    if found_inf:
        return scale / 2, 0, False
    good_steps += 1
    if good_steps == growth_interval:
        return scale * 2, 0, True
    return scale, good_steps, True

In [ ]:
def update_loss_scale(scale, found_inf, good_steps, growth_interval=2000):
    """Dynamic loss scaling. Returns (new_scale, new_good_steps, apply_update)."""
    # ✍️ Your code here
    raise NotImplementedError

In [ ]:
update_loss_scale = practice("update_loss_scale", update_loss_scale, update_loss_scale_reference, cases=[
    (65536.0, True, 37), (65536.0, False, 37), (65536.0, False, 1999), (1024.0, False, 2, 3),
])
scale, good = 2.0 ** 24, 0
for update in range(6):
    _, found_inf = scaled_float16_gradients(layer_grads, scale)
    new_scale, good, applied = update_loss_scale(scale, found_inf, good, growth_interval=3)
    print(f"update {update}: scale 2^{np.log2(scale):.0f}, overflow {found_inf}, update applied {applied}")
    scale = new_scale

### Exercise 2: simulate bfloat16

bfloat16 is float32 with the last 16 bits of the fraction rounded away. Write `to_bfloat16(x)` that rounds a float32 array to the nearest bfloat16 value (ties to even) and returns it as float32. Hints: `bits = x.astype(np.float32).view(np.uint32)`; add `0x7FFF + ((bits >> 16) & 1)` as `np.uint32`; keep the top 16 bits with `& np.uint32(0xFFFF0000)`; `.view(np.float32)` turns the bits back into numbers. Then compare float16 and bfloat16 on `layer_grads` without any loss scaling.

In [ ]:
def to_bfloat16_reference(x):
    bits = np.asarray(x, dtype=np.float32).view(np.uint32)
    rounded = (bits + np.uint32(0x7FFF) + ((bits >> np.uint32(16)) & np.uint32(1))) & np.uint32(0xFFFF0000)
    return rounded.view(np.float32)

In [ ]:
def to_bfloat16(x):
    """Round a float32 array to the nearest bfloat16 value (ties to even); return it as float32."""
    # ✍️ Your code here
    raise NotImplementedError

In [ ]:
to_bfloat16 = practice("to_bfloat16", to_bfloat16, to_bfloat16_reference, cases=[
    (np.array([1.0, 1.001, 1e-8, 3.14159], dtype=np.float32),),
    (np.array([65504.0, 1e30, -2.5e-3], dtype=np.float32),),
], atol=0.0)
for name, grad in zip(layer_names, layer_grads):
    b = to_bfloat16(grad.astype(np.float32)).astype(np.float64)
    print(f"{name:>12}: bfloat16 relative error {np.linalg.norm(b - grad) / np.linalg.norm(grad):.1e}")

### Exercise 3 (question): nested cross-validation

Our search chose a configuration on one validation split and then looked at the test set once. Sketch, as loops in pseudocode, a nested cross-validation of the whole search: an outer loop that holds out each fold in turn and an inner search on the remaining data. What does the outer loop estimate that our single test number does not? What would it cost in trials for 5 outer folds?

### Exercise 4 (question): grouped rows

Suppose each row of the rare-class data came from a customer, with several rows per customer. Which part of the protocol would you change, and what goes wrong in the search if two rows of the same customer land in the training and the validation split?

## Common errors and debugging

- **Storing the weights in float16.** Small updates round away, as the stalled weight showed. Keep a float32 master copy and update that.
- **Loss scaling without unscaling.** The gradients are then $S$ times too large, so the learning rate is effectively multiplied by $S$. Unscale before clipping, logging or updating.
- **Not skipping an update after an overflow.** One `inf` turns the weights into `nan`. Check every gradient for finite values, skip the update and lower the scale.
- **Expecting checkpointing to fix weight or optimiser memory.** It only reduces stored activations.
- **Randomness in recomputed segments.** Dropout masks must be the same in both passes, or the gradients belong to a different network.
- **Timing a single cold run.** The first call pays one-off costs. Repeat, take the best or the median, and compare like with like.
- **Searching learning rates on a linear grid.** Most of the grid then sits at large values. Use a log scale.
- **Keeping only the best trial.** Without the failed and mediocre trials, nobody can judge how sensitive or lucky the winner is.
- **Comparing configurations with different protocols.** Different splits, update budgets or metrics make the ledger meaningless; fix them first.
- **Tuning on the test set.** Every choice above used validation data; the test set was opened once, for the chosen configuration.

## Summary

float16 halves memory but overflows above 65,504, underflows small gradients and rounds away small updates; loss scaling lifts the gradients into range, and a float32 master copy keeps the updates. Checkpointing kept only segment boundaries during the forward pass and rebuilt one segment at a time, reproducing the exact gradients with a fraction of the activation memory and about one extra forward pass. The random search spent a fixed budget under a fixed protocol, recorded every trial, and re-ran the leaders across seeds before a single test evaluation. The memory figures here count NumPy arrays in a tiny network; on real accelerators the same trade-offs appear at far larger scale, alongside allocator overheads that only profiling reveals.

## Next

**04 · Distributed training and the capstone**: split each batch across simulated workers, average their gradients with an all-reduce, check that the result matches single-worker training, make runs reproducible, and finish with the rare-class capstone.